In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy.stats import poisson
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
import xgboost as xgb

In [ ]:
# The team market values here are used as an additional measure of squad strength (helpful especially for promoted teams)
bundesliga_team_values = {
    "2020": {
        "Bayern Munich": 858230000,
        "RB Leipzig": 574950000,
        "Borussia Dortmund": 628400000,
        "Wolfsburg": 256830000,
        "Eintracht Frankfurt": 269150000,
        "Bayer Leverkusen": 373250000,
        "Union Berlin": 82050000,
        "Borussia Monchengladbach": 300750000,
        "Stuttgart": 189250000,
        "Freiburg": 139100000,
        "Hoffenheim": 228150000,
        "Mainz": 163300000,
        "Augsburg": 93050000,
        "Hertha Berlin": 228880000,
        "Arminia Bielefeld": 56730000,
        "Koln": 118200000
    },

    "2021": {
        "Bayern Munich": 790330000,
        "Borussia Dortmund": 558980000,
        "Bayer Leverkusen": 455050000,
        "RB Leipzig": 498950000,
        "Union Berlin": 115900000,
        "Freiburg": 176350000,
        "Koln": 99280000,
        "Mainz": 131030000,
        "Hoffenheim": 211430000,
        "Borussia Monchengladbach": 236800000,
        "Eintracht Frankfurt": 228750000,
        "Wolfsburg": 269300000,
        "Bochum": 52250000,
        "Augsburg": 95300000,
        "Stuttgart": 174230000,
        "Hertha Berlin": 155780000,
        "Arminia Bielefeld": 65850000,
        "Greuther Furth": 36950000
    },
    
    "2022": {
        "Bayern Munich": 948950000,
        "Borussia Dortmund": 567550000,
        "RB Leipzig": 486400000,
        "Union Berlin": 157950000,
        "Freiburg": 200950000,
        "Bayer Leverkusen": 452500000,
        "Eintracht Frankfurt": 324550000,
        "Wolfsburg": 238300000,
        "Mainz": 135900000,
        "Borussia Monchengladbach": 241680000,
        "Koln": 115650000,
        "Hoffenheim": 178800000,
        "Werder Bremen": 89500000,
        "Bochum": 53700000,
        "Augsburg": 132230000,
        "Stuttgart": 151050000,
        "Schalke": 87780000,
        "Hertha Berlin": 94000000
    },
    "2023": {
        "Bayer Leverkusen": 658350000,
        "Stuttgart": 346630000,
        "Bayern Munich": 965150000,
        "RB Leipzig": 545400000,
        "Borussia Dortmund": 494800000,
        "Eintracht Frankfurt": 326650000,
        "Hoffenheim": 176000000,
        "Heidenheim": 66450000,
        "Werder Bremen": 134930000,
        "Freiburg": 180350000,
        "Augsburg": 140800000,
        "Wolfsburg": 206450000,
        "Mainz": 142880000,
        "Borussia Monchengladbach": 165230000,
        "Union Berlin": 160950000,
        "Bochum": 64530000,
        "Koln": 81450000,
        "Darmstadt": 38700000
    },
    "2024": {
        "Bayern Munich": 953550000,
        "Bayer Leverkusen": 623350000,
        "Eintracht Frankfurt": 431250000,
        "Borussia Dortmund": 508280000,
        "Freiburg": 186200000,
        "Mainz": 175130000,
        "RB Leipzig": 513700000,
        "Werder Bremen": 132000000,
        "Stuttgart": 365230000,
        "Borussia Monchengladbach": 187500000,
        "Wolfsburg": 296300000,
        "Augsburg": 158880000,
        "Union Berlin": 136050000,
        "Saint Pauli": 65780000,
        "Hoffenheim": 187100000,
        "Heidenheim": 72530000,
        "Holstein Kiel": 47130000,
        "Bochum": 57100000
    },
    "2025": {
        "Bayern Munich": 994400000,
        "Borussia Dortmund": 520400000,
        "RB Leipzig": 618700000,
        "Stuttgart": 464550000,
        "Hoffenheim": 285800000,
        "Bayer Leverkusen": 548350000,
        "Freiburg": 264850000,
        "Eintracht Frankfurt": 408950000,
        "Augsburg": 180650000,
        "Mainz": 165350000,
        "Union Berlin": 129850000,
        "Borussia Monchengladbach": 171400000,
        "Hamburger SV": 186050000,
        "Koln": 156650000,
        "Werder Bremen": 177830000,
        "Wolfsburg": 234100000,
        "Heidenheim": 78750000,
        "Saint Pauli": 65650000
    }

}

current_market_values = {
        "Bayern Munich": 1070000000,
        "Bayer Leverkusen": 516950000,
        "RB Leipzig": 514600000,
        "Borussia Dortmund": 489200000,
        "Stuttgart": 430450000,
        "Eintracht Frankfurt": 348350000,
        "Hoffenheim": 294950000,
        "Freiburg": 242550000,
        "Mainz": 175200000,
        "Augsburg": 159900000,
        "Werder Bremen": 137830000,
        "Borussia Monchengladbach": 132550000,
        "Koln": 123900000,
        "Union Berlin": 119880000,
        "Hamburger SV": 101150000,
        "Schalke": 67330000,
        "Elversberg": 59100000,
        "Paderborn": 40380000
    }

In [ ]:
# This loads the team match statistics used to construct model features
fb_team_match_stats = pd.read_csv("/Users/tirenioladimeji/Documents/Coursework/bundesliga-2627-predictor/fbref/fbref_team_match_stats.csv")
understat_team_match_stats = pd.read_csv("/Users/tirenioladimeji/Documents/Coursework/bundesliga-2627-predictor/understat/understat_team_match_stats.csv")

master_match_stats = understat_team_match_stats.copy()

fb_home = fb_team_match_stats[
    fb_team_match_stats["venue"] == "Home"
]

fb_away = fb_team_match_stats[
    fb_team_match_stats["venue"] == "Away"
]

formation_cols = [
    "Formation",
    "Opp Formation"
]

for col in formation_cols:
    fb_team_match_stats[col] = (
        fb_team_match_stats[col]
        .astype("string")
    )

fb_home = fb_home[["game_id", "possession", "Formation", "Opp Formation"]].rename(columns={ "possession": "home_possession", "Formation": "home_formation", "Opp Formation": "home_opp_formation"})

fb_away = fb_away[["game_id", "possession", "Formation", "Opp Formation"]].rename(columns={ "possession": "away_possession", "Formation": "away_formation", "Opp Formation": "away_opp_formation"})

master_match_stats = master_match_stats.merge(
    fb_home,
    on="game_id",
    how="left"
)

master_match_stats = master_match_stats.merge(
    fb_away,
    on="game_id",
    how="left"
)

master_match_stats = master_match_stats.sort_values(
    ["date", "game_id"]
).reset_index(drop=True)

master_match_stats.to_csv("master_match_stats.csv", index=False)

/var/folders/8q/7bjjn_zj6wz1001hrcxjjbjr0000gn/T/ipykernel_3963/2605210468.py:1: DtypeWarning: Columns (0,1,2,3,4,5,8,9,12,13,16) have mixed types. Specify dtype option on import or set low_memory=False.
  fb_team_match_stats = pd.read_csv("/Users/tirenioladimeji/Documents/Coursework/bundesliga-2627-predictor/fbref/fbref_team_match_stats.csv")


In [ ]:
# This divides the stats into home and away stats and creates rolling features that describe a team's short and long term performance
# This also includes attacking and defensive strength for each team
master_match_stats = pd.read_csv("master_match_stats.csv")

home = master_match_stats[
    [
        "game_id",
        "date",
        "season_id",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_xg",
        "away_xg",
        "home_np_xg",
        "away_np_xg",
        "home_np_xg_difference",
        "away_np_xg_difference",
        "home_ppda",
        "away_ppda",
        "home_deep_completions",
        "away_deep_completions",
        "home_points",
        "home_expected_points",
    ]
].copy()

home = home.rename(columns={
    "home_team": "team",
    "away_team": "opponent",

    "home_goals": "goals",
    "away_goals": "goals_against",

    "home_xg": "xg",
    "away_xg": "xga",

    "home_np_xg": "np_xg",
    "away_np_xg": "np_xga",

    "home_np_xg_difference": "np_xg_difference",

    "home_ppda": "ppda",

    "home_deep_completions": "deep_completions",

    "home_points": "points",
    "home_expected_points": "expected_points",
})


home["home_or_away"] = "home"

away = master_match_stats[
    [
        "game_id",
        "date",
        "season_id",
        "home_team",
        "away_team",
        "away_goals",
        "home_goals",
        "away_xg",
        "home_xg",
        "away_np_xg",
        "home_np_xg",
        "away_np_xg_difference",
        "home_np_xg_difference",
        "away_ppda",
        "home_ppda",
        "away_deep_completions",
        "home_deep_completions",
        "away_points",
        "away_expected_points",
    ]
].copy()

away = away.rename(columns={
    "away_team": "team",
    "home_team": "opponent",

    "away_goals": "goals",
    "home_goals": "goals_against",

    "away_xg": "xg",
    "home_xg": "xga",

    "away_np_xg": "np_xg",
    "home_np_xg": "np_xga",

    "away_np_xg_difference": "np_xg_difference",

    "away_ppda": "ppda",

    "away_deep_completions": "deep_completions",

    "away_points": "points",
    "away_expected_points": "expected_points",
})


away["home_or_away"] = "away"

team_matches = pd.concat(
    [home, away],
    ignore_index=True
)

team_matches = team_matches[
    [
        "game_id",
        "date",
        "season_id",
        "team",
        "opponent",
        "goals",
        "goals_against",
        "xg",
        "xga",
        "np_xg",
        "np_xga",
        "np_xg_difference",
        "ppda",
        "deep_completions",
        "points",
        "expected_points",
        "home_or_away"
    ]
].copy()

team_matches = team_matches.sort_values(
    ["team", "date", "game_id"]
).reset_index(drop=True)



team_matches["venue"] = team_matches["home_or_away"]



rolling_cols = [
    "xg",
    "xga",
    "goals",
    "goals_against",
    "points"
]

for col in rolling_cols:

    team_matches[f"{col}_last5"] = (
        team_matches
        .groupby("team")[col]
        .transform(
            lambda x: x.shift(1)
                       .rolling(5, min_periods=5)
                       .mean()
        )
    )

    team_matches[f"{col}_last10"] = (
        team_matches
        .groupby("team")[col]
        .transform(
            lambda x: x.shift(1)
                       .rolling(10, min_periods=10)
                       .mean()
        )
    )

    team_matches[f"{col}_season_to_date"] = (
        team_matches
        .groupby(["season_id", "team"])[col]
        .transform(
            lambda x: x.shift(1)
                       .expanding(min_periods=3)
                       .mean()
        )
    )

    team_matches[f"{col}_venue_last5"] = (
        team_matches
        .groupby(["team", "home_or_away"])[col]
        .transform(
            lambda x: x.shift(1)
                       .rolling(5, min_periods=5)
                       .mean()
        )
    )




advanced_cols = [
    "np_xg",
    "np_xga",
    "np_xg_difference",
    "ppda",
    "deep_completions"
]

for col in advanced_cols:

    team_matches[f"{col}_last5"] = (
        team_matches
        .groupby("team")[col]
        .transform(
            lambda x: x.shift(1)
                       .rolling(5, min_periods=5)
                       .mean()
        )
    )

    team_matches[f"{col}_last10"] = (
        team_matches
        .groupby("team")[col]
        .transform(
            lambda x: x.shift(1)
                       .rolling(10, min_periods=10)
                       .mean()
        )
    )

    team_matches[f"{col}_season_to_date"] = (
        team_matches
        .groupby(["season_id", "team"])[col]
        .transform(
            lambda x: x.shift(1)
                       .expanding(min_periods=3)
                       .mean()
        )
    )

    team_matches[f"{col}_venue_last5"] = (
        team_matches
        .groupby(["team", "home_or_away"])[col]
        .transform(
            lambda x: x.shift(1)
                       .rolling(5, min_periods=5)
                       .mean()
        )
    )



home_features = team_matches[
    team_matches["home_or_away"] == "home"
].copy()

away_features = team_matches[
    team_matches["home_or_away"] == "away"
].copy()



home_features = home_features.rename(columns={

    "team": "home_team",
    "opponent": "away_team",

    "xg_last5": "home_xg_last5",
    "xga_last5": "home_xga_last5",
    "goals_last5": "home_goals_last5",
    "goals_against_last5": "home_goals_against_last5",
    "points_last5": "home_points_last5",

    "xg_last10": "home_xg_last10",
    "xga_last10": "home_xga_last10",
    "goals_last10": "home_goals_last10",
    "goals_against_last10": "home_goals_against_last10",
    "points_last10": "home_points_last10",

    "xg_season_to_date": "home_xg_season_to_date",
    "xga_season_to_date": "home_xga_season_to_date",
    "goals_season_to_date": "home_goals_season_to_date",
    "goals_against_season_to_date": "home_goals_against_season_to_date",
    "points_season_to_date": "home_points_season_to_date",


    "xg_venue_last5": "home_xg_venue_last5",
    "xga_venue_last5": "home_xga_venue_last5",
    "goals_venue_last5": "home_goals_venue_last5",
    "goals_against_venue_last5": "home_goals_against_venue_last5",
    "points_venue_last5": "home_points_venue_last5",

    "np_xg_last5": "home_np_xg_last5",
    "np_xga_last5": "home_np_xga_last5",
    "np_xg_difference_last5": "home_np_xg_difference_last5",
    "ppda_last5": "home_ppda_last5",
    "deep_completions_last5": "home_deep_completions_last5",

    "np_xg_last10": "home_np_xg_last10",
    "np_xga_last10": "home_np_xga_last10",
    "np_xg_difference_last10": "home_np_xg_difference_last10",
    "ppda_last10": "home_ppda_last10",
    "deep_completions_last10": "home_deep_completions_last10",

    "np_xg_season_to_date": "home_np_xg_season_to_date",
    "np_xga_season_to_date": "home_np_xga_season_to_date",
    "np_xg_difference_season_to_date": "home_np_xg_difference_season_to_date",
    "ppda_season_to_date": "home_ppda_season_to_date",
    "deep_completions_season_to_date": "home_deep_completions_season_to_date",

    "np_xg_venue_last5": "home_np_xg_venue_last5",
    "np_xga_venue_last5": "home_np_xga_venue_last5",
    "np_xg_difference_venue_last5": "home_np_xg_difference_venue_last5",
    "ppda_venue_last5": "home_ppda_venue_last5",
    "deep_completions_venue_last5": "home_deep_completions_venue_last5",
})




away_features = away_features.rename(columns={

    "team": "away_team",
    "opponent": "home_team",


    "xg_last5": "away_xg_last5",
    "xga_last5": "away_xga_last5",
    "goals_last5": "away_goals_last5",
    "goals_against_last5": "away_goals_against_last5",
    "points_last5": "away_points_last5",

    "xg_last10": "away_xg_last10",
    "xga_last10": "away_xga_last10",
    "goals_last10": "away_goals_last10",
    "goals_against_last10": "away_goals_against_last10",
    "points_last10": "away_points_last10",


    "xg_season_to_date": "away_xg_season_to_date",
    "xga_season_to_date": "away_xga_season_to_date",
    "goals_season_to_date": "away_goals_season_to_date",
    "goals_against_season_to_date": "away_goals_against_season_to_date",
    "points_season_to_date": "away_points_season_to_date",

    "xg_venue_last5": "away_xg_venue_last5",
    "xga_venue_last5": "away_xga_venue_last5",
    "goals_venue_last5": "away_goals_venue_last5",
    "goals_against_venue_last5": "away_goals_against_venue_last5",
    "points_venue_last5": "away_points_venue_last5",

 
    "np_xg_last5": "away_np_xg_last5",
    "np_xga_last5": "away_np_xga_last5",
    "np_xg_difference_last5": "away_np_xg_difference_last5",
    "ppda_last5": "away_ppda_last5",
    "deep_completions_last5": "away_deep_completions_last5",


    "np_xg_last10": "away_np_xg_last10",
    "np_xga_last10": "away_np_xga_last10",
    "np_xg_difference_last10": "away_np_xg_difference_last10",
    "ppda_last10": "away_ppda_last10",
    "deep_completions_last10": "away_deep_completions_last10",

  
    "np_xg_season_to_date": "away_np_xg_season_to_date",
    "np_xga_season_to_date": "away_np_xga_season_to_date",
    "np_xg_difference_season_to_date": "away_np_xg_difference_season_to_date",
    "ppda_season_to_date": "away_ppda_season_to_date",
    "deep_completions_season_to_date": "away_deep_completions_season_to_date",

    "np_xg_venue_last5": "away_np_xg_venue_last5",
    "np_xga_venue_last5": "away_np_xga_venue_last5",
    "np_xg_difference_venue_last5": "away_np_xg_difference_venue_last5",
    "ppda_venue_last5": "away_ppda_venue_last5",
    "deep_completions_venue_last5": "away_deep_completions_venue_last5",
})

merge_keys = [
    "game_id",
    "date",
    "season_id",
    "home_team",
    "away_team"
]


home_feature_cols = [
    col for col in home_features.columns
    if col not in merge_keys
    and col not in [
        "goals",
        "goals_against",
        "xg",
        "xga",
        "np_xg",
        "np_xga",
        "np_xg_difference",
        "ppda",
        "deep_completions",
        "points",
        "expected_points",
        "home_or_away",
        "venue"
    ]
]

away_feature_cols = [
    col for col in away_features.columns
    if col not in merge_keys
    and col not in [
        "goals",
        "goals_against",
        "xg",
        "xga",
        "np_xg",
        "np_xga",
        "np_xg_difference",
        "ppda",
        "deep_completions",
        "points",
        "expected_points",
        "home_or_away",
        "venue"
    ]
]

match_features = home_features[
    merge_keys + home_feature_cols
].merge(
    away_features[
        merge_keys + away_feature_cols
    ],
    on=merge_keys,
    how="inner"
)

match_features["home_xg_advantage"] = (
    match_features["home_xg_last5"]
    - match_features["away_xga_last5"]
)

match_features["away_xg_advantage"] = (
    match_features["away_xg_last5"]
    - match_features["home_xga_last5"]
)

match_features["home_np_xg_advantage"] = (
    match_features["home_np_xg_last5"]
    - match_features["away_np_xga_last5"]
)

match_features["away_np_xg_advantage"] = (
    match_features["away_np_xg_last5"]
    - match_features["home_np_xg_last5"]
)

match_features["home_attack_strength"] = (
    0.6 * match_features["home_xg_last5"]
    + 0.4 * match_features["home_xg_season_to_date"]
)

match_features["away_attack_strength"] = (
    0.6 * match_features["away_xg_last5"]
    + 0.4 * match_features["away_xg_season_to_date"]
)

match_features["home_defensive_strength"] = (
    0.6 * match_features["home_xga_last5"]
    + 0.4 * match_features["home_xga_season_to_date"]
)

match_features["away_defensive_strength"] = (
    0.6 * match_features["away_xga_last5"]
    + 0.4 * match_features["away_xga_season_to_date"]
)

In [ ]:
# This creates prior season stats for both promoted and established Bundesliga teams for any season
season_stats = (
    team_matches
    .groupby(["season_id", "team"])
    .agg(
        matches = ("game_id", "nunique"),
        goals = ("goals", "sum"),
        goals_against = ("goals_against", "sum"),
        points = ("points", "sum")
    ).reset_index()
)

season_stats["goal_difference"] = (
    season_stats["goals"] - season_stats["goals_against"]
)

season_stats["ppg"] = (
    season_stats["points"]
    / season_stats["matches"]
)

season_stats = season_stats.sort_values(
    ["season_id", "points", "goal_difference", "goals"],
    ascending=[True, False, False, False]
)

season_stats["league_position"] = (
    season_stats
    .groupby("season_id")
    .cumcount() + 1
)

season_stats = season_stats.rename(columns={
    "goals": "previous_league_goals",
    "goals_against": "previous_league_goals_against",
    "points": "previous_league_points",
    "ppg": "previous_league_ppg",
    "goal_difference": "previous_league_goal_difference"
})

season_stats["season_id"] = (
    season_stats["season_id"] + 1
)

season_stats["previous_league_level"] = 1
season_stats["promoted"] = 0
season_stats["promotion_rank"] = 0

previous_bundesliga_stats = season_stats.copy()

In [ ]:
# These are the market value-related features used in prediction
def get_market_value(season, team):
    return bundesliga_team_values.get(str(season), {}).get(team)

match_features["home_market_value"] = match_features.apply(
    lambda row: get_market_value(
        row["season_id"],
        row["home_team"]
    ),
    axis=1
)

match_features["away_market_value"] = match_features.apply(
    lambda row: get_market_value(
        row["season_id"],
        row["away_team"]
    ),
    axis=1
)

match_features["home_market_value_previous"] = match_features.apply(
    lambda row: get_market_value(
        row["season_id"] - 1,
        row["home_team"]
    ),
    axis=1
)

match_features["away_market_value_previous"] = match_features.apply(
    lambda row: get_market_value(
        row["season_id"] - 1,
        row["away_team"]
    ),
    axis=1
)

match_features["home_market_value_change"] = (
    match_features["home_market_value"]
    - match_features["home_market_value_previous"]
)

match_features["away_market_value_change"] = (
    match_features["away_market_value"]
    - match_features["away_market_value_previous"]
)


match_features["home_market_value_change_pct"] = np.where(
    match_features["home_market_value_previous"] > 0,
    (
        (
            match_features["home_market_value"]
            - match_features["home_market_value_previous"]
        )
        / match_features["home_market_value_previous"]
    ) * 100,
    np.nan
)

match_features["away_market_value_change_pct"] = np.where(
    match_features["away_market_value_previous"] > 0,
    (
        (
            match_features["away_market_value"]
            - match_features["away_market_value_previous"]
        )
        / match_features["away_market_value_previous"]
    ) * 100,
    np.nan
)


match_features["market_value_change_difference"] = (
    match_features["home_market_value_change_pct"]
    - match_features["away_market_value_change_pct"]
)


match_features["home_log_market_value"] = np.where(
    match_features["home_market_value"] > 0,
    np.log(match_features["home_market_value"]),
    np.nan
)

match_features["away_log_market_value"] = np.where(
    match_features["away_market_value"] > 0,
    np.log(match_features["away_market_value"]),
    np.nan
)


match_features["log_market_value_difference"] = (
    match_features["home_log_market_value"]
    - match_features["away_log_market_value"]
)

In [ ]:
# This loads the list of promoted teams so their first Bundesliga season can be handled appropriately
promoted_teams = pd.read_csv("promoted_teams.csv")


promoted_home = promoted_teams.rename(columns={
    "team": "home_team",
    "promoted": "home_promoted",
    "previous_league_level": "home_previous_league_level",
    "previous_league_position": "home_previous_league_position",
    "previous_league_points": "home_previous_league_points",
    "previous_league_ppg": "home_previous_league_ppg",
    "previous_league_goal_difference": "home_previous_league_goal_difference",
    "previous_league_goals": "home_previous_league_goals",
    "previous_league_goals_against": "home_previous_league_goals_against",
    "previous_league_gf_per_match": "home_previous_league_gf_per_match",
    "previous_league_ga_per_match": "home_previous_league_ga_per_match",
})


promoted_home = promoted_home[
    [
        "season_id",
        "home_team",
        "home_promoted",
        "home_previous_league_level",
        "home_previous_league_position",
        "home_previous_league_points",
        "home_previous_league_ppg",
        "home_previous_league_goal_difference",
        "home_previous_league_goals",
        "home_previous_league_goals_against",
        "home_previous_league_gf_per_match",
        "home_previous_league_ga_per_match",
    ]
].copy()


promoted_away = promoted_teams.rename(columns={
    "team": "away_team",
    "promoted": "away_promoted",
    "previous_league_level": "away_previous_league_level",
    "previous_league_position": "away_previous_league_position",
    "previous_league_points": "away_previous_league_points",
    "previous_league_ppg": "away_previous_league_ppg",
    "previous_league_goal_difference": "away_previous_league_goal_difference",
    "previous_league_goals": "away_previous_league_goals",
    "previous_league_goals_against": "away_previous_league_goals_against",
    "previous_league_gf_per_match": "away_previous_league_gf_per_match",
    "previous_league_ga_per_match": "away_previous_league_ga_per_match",
})


promoted_away = promoted_away[
    [
        "season_id",
        "away_team",
        "away_promoted",
        "away_previous_league_level",
        "away_previous_league_position",
        "away_previous_league_points",
        "away_previous_league_ppg",
        "away_previous_league_goal_difference",
        "away_previous_league_goals",
        "away_previous_league_goals_against",
        "away_previous_league_gf_per_match",
        "away_previous_league_ga_per_match",
    ]
].copy()



match_features = match_features.merge(
    promoted_home,
    on=["season_id", "home_team"],
    how="left"
)


match_features = match_features.merge(
    promoted_away,
    on=["season_id", "away_team"],
    how="left"
)



home_fill_cols = [
    "home_promoted",
    "home_previous_league_level",
    "home_previous_league_position",
    "home_previous_league_points",
    "home_previous_league_ppg",
    "home_previous_league_goal_difference",
    "home_previous_league_goals",
    "home_previous_league_goals_against",
    "home_previous_league_gf_per_match",
    "home_previous_league_ga_per_match",
]

away_fill_cols = [
    "away_promoted",
    "away_previous_league_level",
    "away_previous_league_position",
    "away_previous_league_points",
    "away_previous_league_ppg",
    "away_previous_league_goal_difference",
    "away_previous_league_goals",
    "away_previous_league_goals_against",
    "away_previous_league_gf_per_match",
    "away_previous_league_ga_per_match",
]


match_features[home_fill_cols] = (
    match_features[home_fill_cols].fillna(0)
)

match_features[away_fill_cols] = (
    match_features[away_fill_cols].fillna(0)
)

In [ ]:
# This creates head-to-head features while limiting the history used to avoid over-weighting old results
def get_h2h_features(row, matches, max_matches=5, max_seasons=3):

    current_date = row["date"]
    home_team = row["home_team"]
    away_team = row["away_team"]
    current_season = row["season_id"]


    previous = matches[
        (matches["date"] < current_date)
        &
        (
            (
                (matches["home_team"] == home_team)
                & (matches["away_team"] == away_team)
            )
            |
            (
                (matches["home_team"] == away_team)
                & (matches["away_team"] == home_team)
            )
        )
    ].copy()

    previous = previous[
        previous["season_id"] >= current_season - max_seasons
    ]

    previous = (
        previous
        .sort_values("date", ascending=False)
        .head(max_matches)
    )

    if previous.empty:
        return pd.Series({
            "h2h_matches": 0,
            "h2h_home_wins": np.nan,
            "h2h_draws": np.nan,
            "h2h_away_wins": np.nan,
            "h2h_home_win_rate": np.nan,
            "h2h_draw_rate": np.nan,
            "h2h_away_win_rate": np.nan,
            "h2h_home_goals_avg": np.nan,
            "h2h_away_goals_avg": np.nan,
            "h2h_goal_difference_avg": np.nan
        })

    home_wins = 0
    draws = 0
    away_wins = 0

    home_goals = []
    away_goals = []

    for _, match in previous.iterrows():

        if match["home_team"] == home_team:

            hg = match["home_goals"]
            ag = match["away_goals"]

        else:

            hg = match["away_goals"]
            ag = match["home_goals"]

        home_goals.append(hg)
        away_goals.append(ag)

        if hg > ag:
            home_wins += 1

        elif hg == ag:
            draws += 1

        else:
            away_wins += 1

    n = len(previous)

    return pd.Series({
        "h2h_matches": n,

        "h2h_home_wins": home_wins,
        "h2h_draws": draws,
        "h2h_away_wins": away_wins,

        "h2h_home_win_rate": home_wins / n,
        "h2h_draw_rate": draws / n,
        "h2h_away_win_rate": away_wins / n,

        "h2h_home_goals_avg": np.mean(home_goals),
        "h2h_away_goals_avg": np.mean(away_goals),

        "h2h_goal_difference_avg": (
            np.mean(home_goals) - np.mean(away_goals)
        )
    })


h2h_features = match_features.apply(
    lambda row: get_h2h_features(
        row,
        master_match_stats
    ),
    axis=1
)

match_features = pd.concat(
    [match_features, h2h_features],
    axis=1
)

In [ ]:
# This merges the engineered features with the match outcomes to create the modelling dataset
# The models used for prediction are XGBoost models (using Poisson regression as the objective and negative log-likelihood as the evaluation metric) with time-based cross-validation
# Hyperparameter tuning is employed to search for the best model para,eters for the data
model_data = match_features.merge(
    master_match_stats[
        ["game_id", "home_goals", "away_goals"]
    ],
    on="game_id",
    how="left"
)

target_cols = [
    "home_goals",
    "away_goals"
]

non_feature_cols = [
    "game_id",
    "date",
    "season_id",
    "home_team",
    "away_team",
    *target_cols
]

feature_columns = [
    col
    for col in model_data.columns
    if col not in non_feature_cols
]


train_data = (
    model_data[
        model_data["season_id"].between(2021, 2024)
    ]
    .copy()
)

test_data = (
    model_data[
        model_data["season_id"] == 2025
    ]
    .copy()
)

X_train = train_data[feature_columns]
X_test = test_data[feature_columns]

y_home_train = train_data["home_goals"]
y_home_test = test_data["home_goals"]

y_away_train = train_data["away_goals"]
y_away_test = test_data["away_goals"]


param_distributions = {
    "regressor__n_estimators": [500, 700, 1000],
    "regressor__max_depth": [4, 5, 6, 10],
    "regressor__min_child_weight": [1, 5, 10, 20],
    "regressor__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "regressor__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "regressor__learning_rate": [0.03, 0.05, 0.1],
    "regressor__gamma": [0, 0.1, 0.5, 1],
    "regressor__reg_alpha": [0, 0.1, 0.5, 1],
    "regressor__reg_lambda": [1, 5, 10],
    "regressor__max_delta_step": [0, 1, 5]
}

cv = TimeSeriesSplit(n_splits=3)


home_pipeline = Pipeline([
    (
        "regressor",
        xgb.XGBRegressor(
            objective="count:poisson",
            eval_metric="poisson-nloglik",
            random_state=42
        )
    )
])

away_pipeline = Pipeline([
    (
        "regressor",
        xgb.XGBRegressor(
            objective="count:poisson",
            eval_metric="poisson-nloglik",
            random_state=42
        )
    )
])


home_search = RandomizedSearchCV(
    estimator=home_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

home_search.fit(
    X_train,
    y_home_train
)

home_model = home_search.best_estimator_

print("Best home-goals parameters:")
print(home_search.best_params_)

print(
    "Best home-goals CV MAE:",
    round(-home_search.best_score_, 3)
)


away_search = RandomizedSearchCV(
    estimator=away_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

away_search.fit(
    X_train,
    y_away_train
)

away_model = away_search.best_estimator_

print("Best away-goals parameters:")
print(away_search.best_params_)

print(
    "Best away-goals CV MAE:",
    round(-away_search.best_score_, 3)
)



home_test_pred = np.clip(
    home_model.predict(X_test),
    0,
    None
)

away_test_pred = np.clip(
    away_model.predict(X_test),
    0,
    None
)

test_data["xgb_home_lambda"] = home_test_pred
test_data["xgb_away_lambda"] = away_test_pred


print(
    "2025/26 home-goals test MAE:",
    round(
        mean_absolute_error(
            y_home_test,
            home_test_pred
        ),
        3
    )
)

print(
    "2025/26 away-goals test MAE:",
    round(
        mean_absolute_error(
            y_away_test,
            away_test_pred
        ),
        3
    )
)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best home-goals parameters:
{'regressor__subsample': 0.6, 'regressor__reg_lambda': 10, 'regressor__reg_alpha': 0.1, 'regressor__n_estimators': 500, 'regressor__min_child_weight': 5, 'regressor__max_depth': 10, 'regressor__max_delta_step': 1, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0.5, 'regressor__colsample_bytree': 0.6}
Best home-goals CV MAE: 1.05
Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best away-goals parameters:
{'regressor__subsample': 0.6, 'regressor__reg_lambda': 5, 'regressor__reg_alpha': 0.1, 'regressor__n_estimators': 500, 'regressor__min_child_weight': 1, 'regressor__max_depth': 6, 'regressor__max_delta_step': 1, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0.5, 'regressor__colsample_bytree': 0.6}
Best away-goals CV MAE: 0.943
2025/26 home-goals test MAE: 1.085
2025/26 away-goals test MAE: 0.915


In [ ]:
# This retrains the models on the full set with the best hyperparameters from the previous step
X_full = model_data[feature_columns]

y_home_full = model_data["home_goals"]
y_away_full = model_data["away_goals"]


home_best_params = {
    key.replace("regressor__", ""): value
    for key, value in home_search.best_params_.items()
}

away_best_params = {
    key.replace("regressor__", ""): value
    for key, value in away_search.best_params_.items()
}


final_home_pipeline = Pipeline([
    (
        "regressor",
        xgb.XGBRegressor(
            objective="count:poisson",
            eval_metric="poisson-nloglik",
            random_state=42,
            **home_best_params
        )
    )
])


final_away_pipeline = Pipeline([
    (
        "regressor",
        xgb.XGBRegressor(
            objective="count:poisson",
            eval_metric="poisson-nloglik",
            random_state=42,
            **away_best_params
        )
    )
])


final_home_pipeline.fit(
    X_full,
    y_home_full
)

final_away_pipeline.fit(
    X_full,
    y_away_full
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U36](130,)","['home_xg_last5','home_xg_last10','home_xg_season_to_date',..., 'h2h_home_goals_avg','h2h_away_goals_avg','h2h_goal_difference_avg']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,130
,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'count:poisson'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None


In [ ]:
# This sets up the 2026/27 fixture prediction set
fixtures_2627 = pd.read_csv("/Users/tirenioladimeji/Documents/Coursework/bundesliga-2627-predictor/fbref/fixtures-2627.csv")

prediction_features = fixtures_2627[[
    "week",
    "day",
    "date",
    "time",
    "home_team",
    "away_team",
    "venue"
]].copy()

prediction_features["season_id"] = 2026

prediction_features["home_market_value"] = (
    prediction_features["home_team"].map(current_market_values)
)

prediction_features["away_market_value"] = (
    prediction_features["away_team"].map(current_market_values)
)


prediction_features["home_market_value_previous"] = (
    prediction_features["home_team"].map(bundesliga_team_values["2025"])
)

prediction_features["away_market_value_previous"] = (
    prediction_features["away_team"].map(bundesliga_team_values["2025"])
)

prediction_features["home_market_value_change"] = (
    prediction_features["home_market_value"]
    - prediction_features["home_market_value_previous"]
)

prediction_features["away_market_value_change"] = (
    prediction_features["away_market_value"]
    - prediction_features["away_market_value_previous"]
)

prediction_features["home_market_value_change_pct"] = np.where(
    prediction_features["home_market_value_previous"] > 0,
    (
        prediction_features["home_market_value_change"]
        / prediction_features["home_market_value_previous"]
    ) * 100,
    np.nan
)

prediction_features["away_market_value_change_pct"] = np.where(
    prediction_features["away_market_value_previous"] > 0,
    (
        prediction_features["away_market_value_change"]
        / prediction_features["away_market_value_previous"]
    ) * 100,
    np.nan
)

prediction_features["market_value_change_difference"] = (
    prediction_features["home_market_value_change_pct"]
    - prediction_features["away_market_value_change_pct"]
)

prediction_features["home_log_market_value"] = np.where(
    prediction_features["home_market_value"] > 0,
    np.log(prediction_features["home_market_value"]),
    np.nan
)

prediction_features["away_log_market_value"] = np.where(
    prediction_features["away_market_value"] > 0,
    np.log(prediction_features["away_market_value"]),
    np.nan
)

prediction_features["log_market_value_difference"] = (
    prediction_features["home_log_market_value"]
    - prediction_features["away_log_market_value"]
)

base_stats = [
    "xg",
    "xga",
    "goals",
    "goals_against",
    "points",
    "np_xg",
    "np_xga",
    "np_xg_difference",
    "ppda",
    "deep_completions"
]

form_feature_names = [
    f"{stat}_{window}"
    for stat in base_stats
    for window in [
        "last5",
        "last10",
        "season_to_date",
        "venue_last5"
    ]
]

snapshot_rows = []

for team in team_matches["team"].unique():
    team_history = team_matches[
        team_matches["team"] == team
    ].sort_values(["date", "game_id"])

    for venue in ["home", "away"]:
        venue_history = team_history[
            team_history["home_or_away"] == venue
        ]

        row = {
            "team": team,
            "home_or_away": venue
        }

        for stat in base_stats:
            row[f"{stat}_last5"] = team_history[stat].tail(5).mean()
            row[f"{stat}_last10"] = team_history[stat].tail(10).mean()

            season_2526 = team_history[
                team_history["season_id"] == 2025
            ]
            row[f"{stat}_season_to_date"] = season_2526[stat].mean()

            row[f"{stat}_venue_last5"] = venue_history[stat].tail(5).mean()

        snapshot_rows.append(row)

preseason_form = pd.DataFrame(snapshot_rows)

home_preseason_form = preseason_form[
    preseason_form["home_or_away"] == "home"
].drop(columns="home_or_away").rename(
    columns={
        "team": "home_team",
        **{
            col: f"home_{col}"
            for col in form_feature_names
        }
    }
)

away_preseason_form = preseason_form[
    preseason_form["home_or_away"] == "away"
].drop(columns="home_or_away").rename(
    columns={
        "team": "away_team",
        **{
            col: f"away_{col}"
            for col in form_feature_names
        }
    }
)

prediction_features = prediction_features.merge(
    home_preseason_form,
    on="home_team",
    how="left"
)

prediction_features = prediction_features.merge(
    away_preseason_form,
    on="away_team",
    how="left"
)

prediction_features["home_xg_advantage"] = (
    prediction_features["home_xg_last5"]
    - prediction_features["away_xga_last5"]
)

prediction_features["away_xg_advantage"] = (
    prediction_features["away_xg_last5"]
    - prediction_features["home_xga_last5"]
)

prediction_features["home_np_xg_advantage"] = (
    prediction_features["home_np_xg_last5"]
    - prediction_features["away_np_xga_last5"]
)

prediction_features["away_np_xg_advantage"] = (
    prediction_features["away_np_xg_last5"]
    - prediction_features["home_np_xga_last5"]
)

prediction_features["home_attack_strength"] = (
    0.6 * prediction_features["home_xg_last5"]
    + 0.4 * prediction_features["home_xg_season_to_date"]
)

prediction_features["away_attack_strength"] = (
    0.6 * prediction_features["away_xg_last5"]
    + 0.4 * prediction_features["away_xg_season_to_date"]
)

prediction_features["home_defensive_strength"] = (
    0.6 * prediction_features["home_xga_last5"]
    + 0.4 * prediction_features["home_xga_season_to_date"]
)

prediction_features["away_defensive_strength"] = (
    0.6 * prediction_features["away_xga_last5"]
    + 0.4 * prediction_features["away_xga_season_to_date"]
)


prediction_features = prediction_features.merge(
    promoted_home,
    on=["season_id", "home_team"],
    how="left"
)

prediction_features = prediction_features.merge(
    promoted_away,
    on=["season_id", "away_team"],
    how="left"
)

prediction_features[home_fill_cols] = (
    prediction_features[home_fill_cols].fillna(0)
)

prediction_features[away_fill_cols] = (
    prediction_features[away_fill_cols].fillna(0)
)


h2h_prediction_features = prediction_features.apply(
    lambda row: get_h2h_features(
        row,
        master_match_stats
    ),
    axis=1
)

prediction_features = pd.concat(
    [prediction_features, h2h_prediction_features],
    axis=1
)

In [ ]:
# This selects the same feature set used during training for the 2026/27 fixtures
X_2627 = prediction_features[feature_columns]

xgb_home_model = final_home_pipeline.named_steps["regressor"]
xgb_away_model = final_away_pipeline.named_steps["regressor"]


prediction_features["xgb_home_lambda"] = np.clip(
    xgb_home_model.predict(X_2627),
    0,
    None
)

prediction_features["xgb_away_lambda"] = np.clip(
    xgb_away_model.predict(X_2627),
    0,
    None
)


prediction_features[
    [
        "week",
        "date",
        "home_team",
        "away_team",
        "xgb_home_lambda",
        "xgb_away_lambda"
    ]
].head(30)

,week,date,home_team,away_team,xgb_home_lambda,xgb_away_lambda
0,1,2026-08-28,Bayern Munich,Stuttgart,2.314734,0.911228
1,1,2026-08-29,RB Leipzig,Borussia Monchengladbach,1.911132,0.966949
2,1,2026-08-29,Elversberg,Bayer Leverkusen,1.043951,2.330974
3,1,2026-08-29,Mainz,Paderborn,1.797489,1.267963
4,1,2026-08-29,Union Berlin,Eintracht Frankfurt,1.004740,1.030058
5,1,2026-08-29,Koln,Hoffenheim,1.525452,1.298344
6,1,2026-08-29,Borussia Dortmund,Hamburger SV,2.520466,0.845753
7,1,2026-08-30,Freiburg,Werder Bremen,1.597620,0.705491
8,1,2026-08-30,Augsburg,Schalke,1.641122,1.108123
9,2,2026-09-04,Stuttgart,Koln,2.428909,0.677801


In [ ]:
# This converts a score probability matrix into home-win, draw, and away-win probabilities
def probabilities_from_score_matrix(matrix):

    home_win_probability = 0.0
    draw_probability = 0.0
    away_win_probability = 0.0

    for home_goals in range(matrix.shape[0]):
        for away_goals in range(matrix.shape[1]):

            probability = matrix[
                home_goals,
                away_goals
            ]

            if home_goals > away_goals:
                home_win_probability += probability

            elif home_goals == away_goals:
                draw_probability += probability

            else:
                away_win_probability += probability

    flat_indices = (
        np.argsort(
            matrix.ravel()
        )[::-1]
    )

    top_3 = []

    for index in flat_indices[:3]:

        home_goals, away_goals = np.unravel_index(
            index,
            matrix.shape
        )

        top_3.append({
            "scoreline": f"{home_goals}-{away_goals}",
            "probability": matrix[
                home_goals,
                away_goals
            ]
        })

    return {
        "home_win_probability": home_win_probability,
        "draw_probability": draw_probability,
        "away_win_probability": away_win_probability,

        "home_expected_points": (
            3 * home_win_probability
            + draw_probability
        ),

        "away_expected_points": (
            3 * away_win_probability
            + draw_probability
        ),

        "most_likely_scoreline": top_3[0]["scoreline"],
        "most_likely_scoreline_probability": top_3[0]["probability"],

        "second_likely_scoreline": top_3[1]["scoreline"],
        "second_likely_scoreline_probability": top_3[1]["probability"],

        "third_likely_scoreline": top_3[2]["scoreline"],
        "third_likely_scoreline_probability": top_3[2]["probability"]
    }

In [ ]:
# This is the Dixon-Coles model including the rho correction, model fitting and time decay (to prioritise recent matches)
dc_matches = master_match_stats[
    [
        "game_id",
        "date",
        "season_id",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals"
    ]
].copy()

dc_matches["date"] = pd.to_datetime(dc_matches["date"])

dc_matches = dc_matches.dropna(
    subset=[
        "home_team",
        "away_team",
        "home_goals",
        "away_goals"
    ]
)

dc_matches = dc_matches.sort_values(["date", "game_id"]).reset_index(drop=True)

def dixon_coles_tau(home_goals, away_goals, home_lambda, away_lambda, rho):
    if home_goals == 0 and away_goals == 0:
        return 1 - (home_lambda * away_lambda * rho)
    if home_goals == 0 and away_goals == 1:
        return 1 + (home_lambda * rho)
    if home_goals == 1 and away_goals == 0:
        return 1 + (away_lambda * rho)
    if home_goals == 1 and away_goals == 1:
        return 1 - rho
    return 1.0

def fit_dixon_coles(matches, half_life_days=730):
    matches = matches.copy().sort_values("date").reset_index(drop=True)

    teams = sorted(
        set(matches["home_team"]) |
        set(matches["away_team"])
    )
    n_teams = len(teams)

    team_to_index = {
        team: i
        for i, team in enumerate(teams)
    }

    home_index = matches["home_team"].map(team_to_index).to_numpy()
    away_index = matches["away_team"].map(team_to_index).to_numpy()

    observed_home_goals = matches["home_goals"].astype(int).to_numpy()
    observed_away_goals = matches["away_goals"].astype(int).to_numpy()

    age_in_days = (
        matches["date"].max() - matches["date"]
    ).dt.days.to_numpy()

    time_weights = np.exp(
        -np.log(2) * age_in_days / half_life_days
    )

    def unpack_params(params):
        attack_free = params[:n_teams - 1]
        defence_free = params[n_teams - 1:2 * (n_teams - 1)]

        attack = np.r_[attack_free, -attack_free.sum()]
        defence = np.r_[defence_free, -defence_free.sum()]

        home_advantage = params[-3]
        league_baseline = params[-2]
        rho = params[-1]

        return attack, defence, home_advantage, league_baseline, rho

    def negative_log_likelihood(params):
        attack, defence, home_advantage, league_baseline, rho = unpack_params(params)

        home_lambda = np.exp(
            league_baseline
            + home_advantage
            + attack[home_index]
            + defence[away_index]
        )

        away_lambda = np.exp(
            league_baseline
            + attack[away_index]
            + defence[home_index]
        )

        tau = np.ones(len(matches))

        mask_00 = (observed_home_goals == 0) & (observed_away_goals == 0)
        mask_01 = (observed_home_goals == 0) & (observed_away_goals == 1)
        mask_10 = (observed_home_goals == 1) & (observed_away_goals == 0)
        mask_11 = (observed_home_goals == 1) & (observed_away_goals == 1)

        tau[mask_00] = (
            1 - home_lambda[mask_00] * away_lambda[mask_00] * rho
        )
        tau[mask_01] = 1 + home_lambda[mask_01] * rho
        tau[mask_10] = 1 + away_lambda[mask_10] * rho
        tau[mask_11] = 1 - rho

        if np.any(tau <= 0):
            return 1e12

        probabilities = (
            poisson.pmf(observed_home_goals, home_lambda)
            * poisson.pmf(observed_away_goals, away_lambda)
            * tau
        )

        return -np.sum(
            time_weights
            * np.log(np.clip(probabilities, 1e-12, None))
        )

    initial_baseline = np.log(
        matches[["home_goals", "away_goals"]].to_numpy().mean()
    )

    initial_params = np.r_[
        np.zeros(n_teams - 1),
        np.zeros(n_teams - 1),
        np.log(1.15),
        initial_baseline,
        0.0
    ]

    bounds = (
        [(None, None)] * (2 * (n_teams - 1))
        + [(None, None), (None, None), (-0.2, 0.2)]
    )

    result = minimize(
        negative_log_likelihood,
        initial_params,
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": 2000}
    )

    if not result.success:
        raise RuntimeError(result.message)

    attack, defence, home_advantage, league_baseline, rho = unpack_params(result.x)

    return {
        "teams": teams,
        "attack": dict(zip(teams, attack)),
        "defence": dict(zip(teams, defence)),
        "home_advantage": home_advantage,
        "league_baseline": league_baseline,
        "rho": rho,
        "half_life_days": half_life_days,
        "optimisation_result": result
    }

dc_train = dc_matches[
    dc_matches["season_id"].between(
        2021,
        2024
    )
].copy()

dc_validation_model = fit_dixon_coles(
    dc_train
)

print(
    "Validation DC model fitted on:",
    dc_train["season_id"].min(),
    "-",
    dc_train["season_id"].max()
)


dc_final = fit_dixon_coles(
    dc_matches[
        dc_matches["season_id"].between(
            2021,
            2025
        )
    ].copy()
)

dc_model = dc_final

print(
    "Final DC model fitted on:",
    dc_matches[
        dc_matches["season_id"].between(
            2021,
            2025
        )
    ]["season_id"].min(),
    "-",
    dc_matches[
        dc_matches["season_id"].between(
            2021,
            2025
        )
    ]["season_id"].max()
)

Validation DC model fitted on: 2021 - 2024
Final DC model fitted on: 2021 - 2025


In [ ]:
# This constructs a Dixon-Coles score probability matrix from the expected goals and dependence parameter (rho)
def dixon_coles_score_matrix(home_xg, away_xg, rho, max_goals = 10):
    home_scores = np.arange(max_goals + 1)
    away_scores = np.arange(max_goals + 1)

    matrix = np.outer(
        poisson.pmf(home_scores, home_xg),
        poisson.pmf(away_scores, away_xg)
    )

    for home_goals in range(2):
        for away_goals in range(2):
            matrix[home_goals, away_goals] *= dixon_coles_tau(
                home_goals,
                away_goals,
                home_xg,
                away_xg,
                rho
            )

    matrix = np.clip(matrix, 0, None)

    return matrix / matrix.sum()



In [ ]:
# This prepares historical market-value data for use in the model, and fits linear regression models to establish the relationship between attack/defensive strength and market value
historical_market_values_2025 = (
    bundesliga_team_values["2025"]
)

market_prior_df = pd.DataFrame({
    "team": list(
        historical_market_values_2025.keys()
    ),

    "market_value": list(
        historical_market_values_2025.values()
    )
})

market_prior_df["log_market_value"] = (
    np.log(
        market_prior_df["market_value"]
    )
)

market_prior_df = market_prior_df[
    market_prior_df["team"].isin(
        dc_validation_model["teams"]
    )
].copy()

market_prior_df["dc_attack"] = (
    market_prior_df["team"]
    .map(
        dc_validation_model["attack"]
    )
)

market_prior_df["dc_defence"] = (
    market_prior_df["team"]
    .map(
        dc_validation_model["defence"]
    )
)

market_prior_df = market_prior_df.dropna(
    subset=[
        "log_market_value",
        "dc_attack",
        "dc_defence"
    ]
)


attack_prior_model = LinearRegression()

defence_prior_model = LinearRegression()


attack_prior_model.fit(
    market_prior_df[
        ["log_market_value"]
    ],
    market_prior_df["dc_attack"]
)

defence_prior_model.fit(
    market_prior_df[
        ["log_market_value"]
    ],
    market_prior_df["dc_defence"]
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[-0.07]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['log_market_value']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.256
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


In [ ]:
# This returns the Dixon-Coles attack and defence parameters for a team, including any adjustments required for promoted teams
def get_effective_dc_parameters(team, dc_model):
    
    if team in dc_model["attack"]:
        return (
            dc_model["attack"][team],
            dc_model["defence"][team]
        )

    
    market_value = current_market_values.get(team)
    
    if market_value is None:
        return 0.0, 0.0
    
    log_market_value = np.log(market_value)
    
    attack = attack_prior_model.predict(
        pd.DataFrame({
            "log_market_value": [log_market_value]
        })
    )[0]
    
    defence = defence_prior_model.predict(
        pd.DataFrame({
            "log_market_value": [log_market_value]
        })
    )[0]
    
    return attack, defence


def get_dc_expected_goals(home_team, away_team, dc_model):

    home_attack, home_defence = (
        get_effective_dc_parameters(
            home_team,
            dc_model
        )
    )

    away_attack, away_defence = (
        get_effective_dc_parameters(
            away_team,
            dc_model
        )
    )

    home_lambda = np.exp(
        dc_model["league_baseline"]
        + dc_model["home_advantage"]
        + home_attack
        + away_defence
    )

    away_lambda = np.exp(
        dc_model["league_baseline"]
        + away_attack
        + home_defence
    )

    return home_lambda, away_lambda

In [ ]:
# This generates Dixon-Coles predictions on the held-out 2025/26 test season
test_dc_predictions = test_data.apply(
    lambda row: get_dc_expected_goals(
        row["home_team"],
        row["away_team"],
        dc_validation_model
    ),
    axis=1,
    result_type="expand"
)

test_dc_predictions.columns = [
    "dc_home_lambda",
    "dc_away_lambda"
]

test_data = pd.concat(
    [
        test_data.reset_index(drop=True),
        test_dc_predictions.reset_index(drop=True)
    ],
    axis=1
)



dc_2627_predictions = prediction_features.apply(
    lambda row: get_dc_expected_goals(
        row["home_team"],
        row["away_team"],
        dc_final
    ),
    axis=1,
    result_type="expand"
)

dc_2627_predictions.columns = [
    "dc_home_lambda",
    "dc_away_lambda"
]

prediction_features = pd.concat(
    [
        prediction_features.reset_index(drop=True),
        dc_2627_predictions.reset_index(drop=True)
    ],
    axis=1
)


print(
    prediction_features[
        [
            "home_team",
            "away_team",
            "xgb_home_lambda",
            "xgb_away_lambda",
            "dc_home_lambda",
            "dc_away_lambda"
        ]
    ].head(20)
)

                   home_team                 away_team  xgb_home_lambda  \
0              Bayern Munich                 Stuttgart         2.314734   
1                 RB Leipzig  Borussia Monchengladbach         1.911132   
2                 Elversberg          Bayer Leverkusen         1.043951   
3                      Mainz                 Paderborn         1.797489   
4               Union Berlin       Eintracht Frankfurt         1.004740   
5                       Koln                Hoffenheim         1.525452   
6          Borussia Dortmund              Hamburger SV         2.520466   
7                   Freiburg             Werder Bremen         1.597620   
8                   Augsburg                   Schalke         1.641122   
9                  Stuttgart                      Koln         2.428909   
10                Hoffenheim         Borussia Dortmund         1.878077   
11             Werder Bremen                RB Leipzig         1.075987   
12          Bayer Leverku

In [ ]:
# These are the Dixon-Coles evaluation results
print(
    "2025/26 DC home MAE:",
    round(
        mean_absolute_error(
            test_data["home_goals"],
            test_data["dc_home_lambda"]
        ),
        3
    )
)

print(
    "2025/26 XGBoost home MAE:",
    round(
        mean_absolute_error(
            test_data["home_goals"],
            test_data["xgb_home_lambda"]
        ),
        3
    )
)

print(
    "2025/26 DC away MAE:",
    round(
        mean_absolute_error(
            test_data["away_goals"],
            test_data["dc_away_lambda"]
        ),
        3
    )
)

print(
    "2025/26 XGBoost away MAE:",
    round(
        mean_absolute_error(
            test_data["away_goals"],
            test_data["xgb_away_lambda"]
        ),
        3
    )
)

2025/26 DC home MAE: 1.037
2025/26 XGBoost home MAE: 1.085
2025/26 DC away MAE: 0.911
2025/26 XGBoost away MAE: 0.915


In [ ]:
# These are the XGBoost evaluation results
print(
    "DC home MAE:",
    mean_absolute_error(
        test_data["home_goals"],
        test_data["dc_home_lambda"]
    )
)

print(
    "XGBoost home MAE:",
    mean_absolute_error(
        test_data["home_goals"],
        test_data["xgb_home_lambda"]
    )
)

print(
    "DC away MAE:",
    mean_absolute_error(
        test_data["away_goals"],
        test_data["dc_away_lambda"]
    )
)

print(
    "XGBoost away MAE:",
    mean_absolute_error(
        test_data["away_goals"],
        test_data["xgb_away_lambda"]
    )
)

DC home MAE: 1.0372117063005246
XGBoost home MAE: 1.0853593349456787
DC away MAE: 0.9109449151606529
XGBoost away MAE: 0.9146644473075867


In [ ]:
# This evaluates different weights for combining Dixon-Coles and XGBoost expected-goal predictions
blend_results = []

for dc_weight in np.arange(0, 1.01, 0.05):

    xgb_weight = 1 - dc_weight

    blended_home = (
        dc_weight * test_data["dc_home_lambda"]
        + xgb_weight * test_data["xgb_home_lambda"]
    )

    blended_away = (
        dc_weight * test_data["dc_away_lambda"]
        + xgb_weight * test_data["xgb_away_lambda"]
    )

    home_mae = mean_absolute_error(
        test_data["home_goals"],
        blended_home
    )

    away_mae = mean_absolute_error(
        test_data["away_goals"],
        blended_away
    )

    overall_mae = (home_mae + away_mae) / 2

    blend_results.append({
        "dc_weight": dc_weight,
        "xgb_weight": xgb_weight,
        "home_mae": home_mae,
        "away_mae": away_mae,
        "overall_mae": overall_mae
    })

blend_results = pd.DataFrame(blend_results)

blend_results.sort_values(
    "overall_mae"
).head(10)

,dc_weight,xgb_weight,home_mae,away_mae,overall_mae
17,0.85,0.15,1.041377,0.905925,0.973651
16,0.80,0.20,1.042840,0.904558,0.973699
18,0.90,0.10,1.039962,0.907529,0.973746
15,0.75,0.25,1.044335,0.903384,0.973860
19,0.95,0.05,1.038559,0.909197,0.973878
20,1.00,0.00,1.037212,0.910945,0.974078
14,0.70,0.30,1.045873,0.902437,0.974155
13,0.65,0.35,1.047522,0.901690,0.974606
12,0.60,0.40,1.049597,0.901074,0.975335
11,0.55,0.45,1.051770,0.900743,0.976257


In [ ]:
# This selects the blend that achieves the lowest validation error
best_blend = (
    blend_results
    .sort_values("overall_mae")
    .iloc[0]
)

dc_weight = best_blend["dc_weight"]
xgb_weight = best_blend["xgb_weight"]

print("Best DC weight:", dc_weight)
print("Best XGBoost weight:", xgb_weight)

Best DC weight: 0.8500000000000001
Best XGBoost weight: 0.1499999999999999


In [ ]:
# This uses the model blend to perform the 2026/27 season simulations
prediction_features["blended_home_xg"] = (
    dc_weight
    * prediction_features["dc_home_lambda"]
    +
    xgb_weight
    * prediction_features["xgb_home_lambda"]
)

prediction_features["blended_away_xg"] = (
    dc_weight
    * prediction_features["dc_away_lambda"]
    +
    xgb_weight
    * prediction_features["xgb_away_lambda"]
)

fixture_matrices = [

    dixon_coles_score_matrix(
        row.blended_home_xg,
        row.blended_away_xg,
        dc_final["rho"],
        max_goals=10
    )

    for row
    in prediction_features.itertuples()
]

fixture_probability_records = []

for matrix in fixture_matrices:

    probabilities = (
        probabilities_from_score_matrix(
            matrix
        )
    )

    fixture_probability_records.append(
        probabilities
    )


fixture_probabilities = pd.DataFrame(
    fixture_probability_records
)


prediction_features = pd.concat(
    [
        prediction_features.reset_index(drop=True),
        fixture_probabilities.reset_index(drop=True)
    ],
    axis=1
)


prediction_features[
    [
        "week",
        "date",
        "home_team",
        "away_team",

        "xgb_home_lambda",
        "xgb_away_lambda",

        "dc_home_lambda",
        "dc_away_lambda",

        "blended_home_xg",
        "blended_away_xg",

        "home_win_probability",
        "draw_probability",
        "away_win_probability",

        "most_likely_scoreline",
        "most_likely_scoreline_probability",

        "second_likely_scoreline",
        "second_likely_scoreline_probability",

        "third_likely_scoreline",
        "third_likely_scoreline_probability"
    ]
].head(30)

,week,date,home_team,away_team,xgb_home_lambda,xgb_away_lambda,dc_home_lambda,dc_away_lambda,blended_home_xg,blended_away_xg,home_win_probability,draw_probability,away_win_probability,most_likely_scoreline,most_likely_scoreline_probability,second_likely_scoreline,second_likely_scoreline_probability,third_likely_scoreline,third_likely_scoreline_probability
0,1,2026-08-28,Bayern Munich,Stuttgart,2.314734,0.911228,3.099562,1.248564,2.981837,1.197964,0.729144,0.153797,0.117059,2-1,0.081515,3-1,0.081022,2-0,0.068045
1,1,2026-08-29,RB Leipzig,Borussia Monchengladbach,1.911132,0.966949,2.213164,1.115514,2.167859,1.093230,0.609583,0.217190,0.173227,1-1,0.099693,2-1,0.098510,2-0,0.090109
2,1,2026-08-29,Elversberg,Bayer Leverkusen,1.043951,2.330974,0.801617,1.986301,0.837967,2.038002,0.135480,0.221064,0.643456,0-2,0.117049,1-1,0.105585,0-1,0.105534
3,1,2026-08-29,Mainz,Paderborn,1.797489,1.267963,1.665256,0.658645,1.685091,0.750043,0.586857,0.260048,0.153095,1-0,0.136859,2-0,0.124352,1-1,0.121431
4,1,2026-08-29,Union Berlin,Eintracht Frankfurt,1.004740,1.030058,1.357124,1.487167,1.304267,1.418601,0.333765,0.280001,0.386235,1-1,0.133317,1-2,0.086205,0-1,0.081400
5,1,2026-08-29,Koln,Hoffenheim,1.525452,1.298344,1.612274,1.659085,1.599250,1.604974,0.371631,0.254295,0.374074,1-1,0.114286,1-2,0.083608,2-1,0.083310
6,1,2026-08-29,Borussia Dortmund,Hamburger SV,2.520466,0.845753,2.324878,0.884466,2.354216,0.878659,0.694667,0.190119,0.115214,2-0,0.109310,2-1,0.096046,1-1,0.089505
7,1,2026-08-30,Freiburg,Werder Bremen,1.597620,0.705491,1.724697,1.196714,1.705635,1.123031,0.499003,0.261459,0.239537,1-1,0.124162,2-1,0.096530,1-0,0.089816
8,1,2026-08-30,Augsburg,Schalke,1.641122,1.108123,1.784129,1.019980,1.762678,1.033202,0.534012,0.255961,0.210027,1-1,0.121986,2-1,0.098009,1-0,0.096850
9,2,2026-09-04,Stuttgart,Koln,2.428909,0.677801,2.264541,1.098918,2.289196,1.035751,0.646725,0.203870,0.149405,2-1,0.097633,2-0,0.094263,1-1,0.093569


In [ ]:
# This performs 10000 Monte Carlo season simulations using a reproducible RNG
N_SIMULATIONS = 10000

rng = np.random.default_rng(42)

def simulate_season(fixtures, score_matrices, rng):
    teams = sorted(
        set(fixtures["home_team"])
        |
        set(fixtures["away_team"])
    )

    table = {
        team: {
            "team": team,
            "points": 0,
            "goals_for": 0,
            "goals_against": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0
        }
        for team in teams
    }

    match_results = []

    for fixture, matrix in zip(
        fixtures[
            [
                "home_team",
                "away_team"
            ]
        ].itertuples(index=False),
        score_matrices
    ):

        home_team, away_team = fixture

        outcome_index = rng.choice(
            matrix.size,
            p=matrix.ravel()
        )

        home_goals, away_goals = np.unravel_index(
            outcome_index,
            matrix.shape
        )

        match_results.append({
            "home_team": home_team,
            "away_team": away_team,
            "home_goals": home_goals,
            "away_goals": away_goals
        })

        table[home_team]["goals_for"] += home_goals
        table[home_team]["goals_against"] += away_goals

        table[away_team]["goals_for"] += away_goals
        table[away_team]["goals_against"] += home_goals

        if home_goals > away_goals:

            table[home_team]["points"] += 3
            table[home_team]["wins"] += 1
            table[away_team]["losses"] += 1

        elif home_goals < away_goals:

            table[away_team]["points"] += 3
            table[away_team]["wins"] += 1
            table[home_team]["losses"] += 1

        else:

            table[home_team]["points"] += 1
            table[away_team]["points"] += 1
            table[home_team]["draws"] += 1
            table[away_team]["draws"] += 1

    table = pd.DataFrame(table.values())

    table["goal_difference"] = (
        table["goals_for"]
        -
        table["goals_against"]
    )

    table = (
        table
        .sort_values(
            [
                "points",
                "goal_difference",
                "goals_for"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    table["position"] = np.arange(
        1,
        len(table) + 1
    )

    return table, match_results


simulated_tables = []
simulated_matches = []

for simulation in range(N_SIMULATIONS):

    season_table, match_results = simulate_season(
        prediction_features,
        fixture_matrices,
        rng
    )

    season_table["simulation"] = simulation

    simulated_tables.append(
        season_table
    )

    for match in match_results:
        match["simulation"] = simulation
        simulated_matches.append(match)


simulated_tables = pd.concat(
    simulated_tables,
    ignore_index=True
)

simulated_matches = pd.DataFrame(
    simulated_matches
)

In [ ]:
# This summarises the simulated score distributions to obtain the most likely scoreline and expected goals for each fixture
def get_predicted_scorelines(simulated_matches):

    results = []

    for (home_team, away_team), group in simulated_matches.groupby(
        ["home_team", "away_team"]
    ):

        scores = list(
            zip(
                group["home_goals"],
                group["away_goals"]
            )
        )

        score_counts = Counter(scores)

        top_scores = score_counts.most_common(3)

        average_home_goals = group["home_goals"].mean()
        average_away_goals = group["away_goals"].mean()

        results.append({
            "home_team": home_team,
            "away_team": away_team,

            "predicted_score": (
                f"{top_scores[0][0][0]}"
                f"-"
                f"{top_scores[0][0][1]}"
            ),

            "score_probability": (
                top_scores[0][1] /
                len(group)
            ),

            "second_most_likely_score": (
                f"{top_scores[1][0][0]}"
                f"-"
                f"{top_scores[1][0][1]}"
            ),

            "second_score_probability": (
                top_scores[1][1] /
                len(group)
            ),

            "third_most_likely_score": (
                f"{top_scores[2][0][0]}"
                f"-"
                f"{top_scores[2][0][1]}"
            ),

            "third_score_probability": (
                top_scores[2][1] /
                len(group)
            ),

            "expected_home_goals": average_home_goals,
            "expected_away_goals": average_away_goals
        })

    return pd.DataFrame(results)

predicted_scorelines = get_predicted_scorelines(simulated_matches)

predicted_scorelines[
    [
        "home_team",
        "away_team",
        "predicted_score",
        "score_probability",
        "second_most_likely_score",
        "second_score_probability",
        "third_most_likely_score",
        "third_score_probability",
        "expected_home_goals",
        "expected_away_goals"
    ]
].round(3).head(30)



,home_team,away_team,predicted_score,score_probability,second_most_likely_score,second_score_probability,third_most_likely_score,third_score_probability,expected_home_goals,expected_away_goals
0,Augsburg,Bayer Leverkusen,1-1,0.112,1-2,0.105,0-1,0.081,1.144,1.916
1,Augsburg,Bayern Munich,0-2,0.088,1-2,0.087,0-3,0.081,0.994,2.806
2,Augsburg,Borussia Dortmund,1-1,0.109,1-2,0.093,0-1,0.085,1.102,1.971
3,Augsburg,Borussia Monchengladbach,1-1,0.127,2-1,0.090,1-2,0.080,1.508,1.342
4,Augsburg,Eintracht Frankfurt,1-1,0.127,1-2,0.090,2-1,0.077,1.356,1.552
5,Augsburg,Elversberg,1-1,0.137,1-0,0.132,0-0,0.110,1.440,0.891
6,Augsburg,Freiburg,1-1,0.132,1-2,0.086,2-1,0.084,1.392,1.427
7,Augsburg,Hamburger SV,1-1,0.136,1-0,0.093,0-0,0.089,1.447,1.138
8,Augsburg,Hoffenheim,1-1,0.124,1-2,0.083,2-1,0.082,1.549,1.590
9,Augsburg,Koln,1-1,0.135,2-1,0.097,1-0,0.087,1.529,1.199


In [ ]:
# This aggregates the Monte Carlo simulations into expected league positions, points, and competition/relegation probabilities.
season_summary = (
    simulated_tables
    .groupby("team")
    .agg(
        expected_wins=("wins", "mean"),
        expected_draws=("draws", "mean"),
        expected_losses=("losses", "mean"),

        expected_goals_for=(
            "goals_for",
            "mean"
        ),

        expected_goals_against=(
            "goals_against",
            "mean"
        ),

        expected_goal_difference=(
            "goal_difference",
            "mean"
        ),

        expected_points=(
            "points",
            "mean"
        ),

        expected_position=(
            "position",
            "mean"
        ),

        title_probability=(
            "position",
            lambda x: (x == 1).mean()
        ),

        top_four_probability=(
            "position",
            lambda x: (x <= 4).mean()
        ),

        relegation_playoff_probability=(
            "position",
            lambda x: (x == 16).mean()
        ),

        relegation_probability=(
            "position",
            lambda x: (x >= 17).mean()
        )
    )
    .assign(
        played=lambda df:
            (
                df["expected_wins"]
                + df["expected_draws"]
                + df["expected_losses"]
            )
    )
    .sort_values(
        [
            "expected_points",
            "expected_goal_difference",
            "expected_goals_for"
        ],
        ascending=False
    )
    .reset_index()
)


season_summary["expected_position"] = (
    season_summary["expected_position"]
    .round(2)
)

season_summary.index += 1

season_summary[
    [
        "team",
        "played",
        "expected_wins",
        "expected_draws",
        "expected_losses",
        "expected_goals_for",
        "expected_goals_against",
        "expected_goal_difference",
        "expected_points",
        "expected_position",
        "title_probability",
        "top_four_probability",
        "relegation_playoff_probability",
        "relegation_probability"
    ]
].round(3)


,team,played,expected_wins,expected_draws,expected_losses,expected_goals_for,expected_goals_against,expected_goal_difference,expected_points,expected_position,title_probability,top_four_probability,relegation_playoff_probability,relegation_probability
1,Bayern Munich,34.0,25.360,4.868,3.772,101.600,35.866,65.733,80.947,1.12,0.904,1.000,0.000,0.000
2,Borussia Dortmund,34.0,19.295,7.231,7.474,71.460,41.685,29.776,65.116,3.23,0.043,0.829,0.000,0.000
3,Bayer Leverkusen,34.0,19.194,7.336,7.469,70.545,41.058,29.487,64.919,3.25,0.038,0.824,0.000,0.000
4,RB Leipzig,34.0,17.309,7.784,8.907,64.110,43.888,20.221,59.712,4.49,0.009,0.577,0.000,0.000
5,Stuttgart,34.0,16.762,7.668,9.570,64.262,47.118,17.144,57.954,4.99,0.006,0.459,0.000,0.000
6,Eintracht Frankfurt,34.0,14.128,7.993,11.880,57.016,52.804,4.212,50.376,7.54,0.001,0.123,0.006,0.002
7,Hoffenheim,34.0,13.085,7.911,13.005,55.775,57.219,-1.444,47.165,8.81,0.000,0.055,0.013,0.006
8,Mainz,34.0,12.495,8.829,12.676,47.710,49.308,-1.598,46.313,9.14,0.000,0.045,0.016,0.009
9,Freiburg,34.0,12.498,8.355,13.146,50.017,53.143,-3.126,45.850,9.38,0.000,0.040,0.020,0.011
10,Borussia Monchengladbach,34.0,11.636,8.272,14.092,48.828,56.258,-7.431,43.181,10.52,0.000,0.017,0.035,0.024
